# EoMT inference and evaluation on Cityscapes — Step 4

Comparison the two pre-trained EoMT models (one trained on COCO panoptic, one on Cityscapes semantic) on the Cityscapes validation set.

- **Qualitative**: visualize predictions on a sample image. The Cityscapes model is visualized as semantic segmentation; the COCO model as a panoptic prediction (following the brief).
- **Quantitative**: evaluate semantic mIoU on the full validation set for both models. Since the two models predict different class spaces (19 Cityscapes classes vs. 133 COCO panoptic classes), the COCO model's predictions are first mapped to the Cityscapes label space via a hand-defined many-to-one mapping; the same evaluation pipeline is then applied to both.

To switch between the two models, change `TASK` in the setup cell (`"coco"` or `"cityscapes"`) and re-run the notebook from the top.


## Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
%cd /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/eomt


In [ ]:
!pip install -q -r requirements.txt


## Imports and configuration

Set `TASK` to `"cityscapes"` or `"coco"` to select which pre-trained model to evaluate.


In [ ]:
import importlib
import warnings

import yaml
import numpy as np
import torch
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
from lightning import seed_everything

seed_everything(0, verbose=False)

# Which model to evaluate: "cityscapes" or "coco".
TASK = "coco"

device  = 0
img_idx = 0  # Validation index used for the qualitative visualization.
data_path = "/content/drive/MyDrive/Fundamentals_Progetto/Coding_Part_Project"

# Cityscapes config — also used to load the val dataloader for evaluation.
config_path = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# COCO config — only needed to build the COCO model with the correct
# (133-class, panoptic) hyperparameters.
coco_config_path = "configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"
with open(coco_config_path, "r") as f:
    coco_config = yaml.safe_load(f)


def create_mapping(images, ignore_index):
    """Build a class-id -> RGB color map covering all ids present in `images`."""
    unique_ids = np.unique(np.concatenate([np.unique(img) for img in images]))
    valid_ids = unique_ids[unique_ids != ignore_index]
    colors = np.array(
        [plt.cm.hsv(i / max(len(valid_ids), 1))[:3] for i in range(len(valid_ids))]
    )
    mapping = {cid: colors[i] for i, cid in enumerate(valid_ids)}
    mapping[ignore_index] = np.array([0, 0, 0])
    return mapping


def apply_colormap(image, mapping):
    colored_image = np.zeros((*image.shape, 3))
    for cid in np.unique(image):
        colored_image[image == cid] = mapping.get(cid, [0, 0, 0])
    return colored_image


## Dataset (Cityscapes validation)


In [ ]:
# Cityscapes validation dataloader — used for both qualitative inspection
# (Cityscapes branch) and quantitative mIoU on both models.
data_module_name, class_name = config["data"]["class_path"].rsplit(".", 1)
data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = config["data"].get("init_args", {})

data = data_module(
    path=data_path,
    batch_size=1,
    num_workers=0,
    check_empty_targets=False,
    **data_module_kwargs,
).setup()
print(f"Cityscapes val samples: {len(data.val_dataloader().dataset)}")


## Build the model selected by `TASK`


In [ ]:
# Build the model selected by TASK.
# Note: the model architecture itself (encoder + decoder) is the same for both;
# only the classification head's output dimension differs (19 vs 133).
warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module`.*",
)

active_config = config if TASK == "cityscapes" else coco_config

# For the COCO model we need a temporary COCO data module just to read its
# num_classes / img_size (the COCO val set itself is not used).
if TASK == "coco":
    coco_dm_name, coco_dm_cls = coco_config["data"]["class_path"].rsplit(".", 1)
    coco_data_cls = getattr(importlib.import_module(coco_dm_name), coco_dm_cls)
    coco_data_kwargs = coco_config["data"].get("init_args", {})
    model_data = coco_data_cls(
        path=data_path, batch_size=1, num_workers=0,
        check_empty_targets=False, **coco_data_kwargs,
    )
else:
    model_data = data

encoder_cfg = active_config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=model_data.img_size, **encoder_cfg.get("init_args", {}))

network_cfg = active_config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
network = network_cls(
    masked_attn_enabled=False,
    num_classes=model_data.num_classes,
    encoder=encoder,
    **network_kwargs,
)

lit_module_name, lit_class_name = active_config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {k: v for k, v in active_config["model"]["init_args"].items() if k != "network"}
if "stuff_classes" in active_config["data"].get("init_args", {}):
    model_kwargs["stuff_classes"] = active_config["data"]["init_args"]["stuff_classes"]

model = lit_cls(
    img_size=model_data.img_size,
    num_classes=model_data.num_classes,
    network=network,
    **model_kwargs,
).eval().to(f"cuda:{device}")

print(f"Model built: TASK={TASK}, img_size={model_data.img_size}, num_classes={model_data.num_classes}")


## Load pre-trained weights


In [ ]:
# Load the pre-trained weights matching the selected TASK.
WEIGHTS = {
    "cityscapes": f"{data_path}/CourseProjectAnomaly/eomt_cityscapes.bin",
    "coco":       f"{data_path}/CourseProjectAnomaly/eomt_coco.bin",
}

state_dict = torch.load(WEIGHTS[TASK], map_location=f"cuda:{device}", weights_only=True)
model.load_state_dict(state_dict, strict=False)
print(f"Loaded {TASK} weights from {WEIGHTS[TASK]}")


## COCO -> Cityscapes class mapping

Needed only when `TASK = "coco"`. Defined unconditionally so the notebook is self-contained for either task.


In [ ]:
# COCO panoptic class index (0-based) -> Cityscapes train id.
# Cityscapes classes:
#   0  road        5  pole          10 sky        15 bus
#   1  sidewalk    6  traffic light 11 person     16 train
#   2  building    7  traffic sign  12 rider      17 motorcycle
#   3  wall        8  vegetation    13 car        18 bicycle
#   4  fence       9  terrain       14 truck
#
# COCO classes without a sensible Cityscapes counterpart map to VOID and are
# excluded from the mIoU computation (consistent with how Cityscapes
# treats its own ignore label).
COCO_TO_CITYSCAPES = {
    # THINGS
    0:  11,  # person
    1:  18,  # bicycle
    2:  13,  # car
    3:  17,  # motorcycle
    5:  15,  # bus
    6:  16,  # train
    7:  14,  # truck
    9:   6,  # traffic light
    10:  5,  # fire hydrant   -> pole
    11:  7,  # stop sign      -> traffic sign
    92:  5,  # light          -> pole

    # STUFF
    82:  2,  # bridge         -> building
    86:  2,  # door-stuff     -> building
    88:  8,  # flower         -> vegetation
    90:  9,  # gravel         -> terrain
    91:  2,  # house          -> building
    94:  4,  # net            -> fence
    96:  9,  # platform       -> terrain
    97:  9,  # playingfield   -> terrain
    98:  9,  # railroad       -> terrain
    100: 0,  # road
    101: 2,  # roof           -> building
    102: 9,  # sand           -> terrain
    105: 9,  # snow           -> terrain
    107: 2,  # tent           -> building
    109: 3,  # wall-brick     -> wall
    110: 3,  # wall-stone     -> wall
    111: 3,  # wall-tile      -> wall
    112: 3,  # wall-wood      -> wall
    114: 2,  # window-blind   -> building
    115: 2,  # window-other   -> building
    116: 8,  # tree-merged    -> vegetation
    117: 4,  # fence-merged
    118: 2,  # ceiling-merged -> building
    119: 10, # sky-other-merged -> sky
    123: 1,  # pavement-merged -> sidewalk
    125: 9,  # grass-merged   -> terrain
    126: 9,  # dirt-merged    -> terrain
    129: 2,  # building-other-merged
    130: 9,  # rock-merged    -> terrain
    131: 3,  # wall-other-merged -> wall
}
VOID = 255  # all other COCO classes -> ignored during evaluation


def map_coco_to_cityscapes(pred_coco):
    """Discrete (argmax) mapping from a COCO prediction array to Cityscapes ids."""
    out = np.full_like(pred_coco, VOID)
    for coco_idx, city_idx in COCO_TO_CITYSCAPES.items():
        out[pred_coco == coco_idx] = city_idx
    return out


def coco_logits_to_cityscapes_logits(logits_coco, num_cs=19):
    """Sum-aggregate per-pixel COCO logits into Cityscapes class scores.

    Multiple COCO classes mapped to the same Cityscapes class have their
    (already-normalized) EoMT scores summed; the argmax is then taken in the
    19-class space. This is logit-level mapping rather than argmax-level,
    which keeps information from sibling COCO classes (e.g. all wall-* COCO
    classes contribute to the Cityscapes 'wall' score).
    """
    scores_cs = torch.zeros(
        num_cs, *logits_coco.shape[1:],
        device=logits_coco.device, dtype=logits_coco.dtype,
    )
    for coco_idx, cs_idx in COCO_TO_CITYSCAPES.items():
        scores_cs[cs_idx] += logits_coco[coco_idx]
    return scores_cs


CITYSCAPES_NAMES = [
    "road", "sidewalk", "building", "wall", "fence", "pole",
    "traffic light", "traffic sign", "vegetation", "terrain",
    "sky", "person", "rider", "car", "truck", "bus", "train",
    "motorcycle", "bicycle",
]


## Inference functions

- `infer_semantic` — Cityscapes model, direct semantic output.
- `infer_semantic_from_coco` — COCO model, with logit-level mapping to the 19-class space.
- `infer_panoptic` — COCO model in panoptic mode (used only for visualization).


In [ ]:
IGNORE_INDEX = 255


def infer_semantic(img, target):
    """Semantic inference for the Cityscapes model (19-class output)."""
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(f"cuda:{device}")]
        img_sizes = [img.shape[-2:] for img in imgs]
        crops, origins = model.window_imgs_semantic(imgs)

        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], model.img_size, mode="bilinear"
        )
        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        preds = logits[0].argmax(0).cpu()

    pred_array = preds.numpy()
    target_array = model.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[0].numpy()
    return pred_array, target_array


def infer_semantic_from_coco(img, tta_hflip=False):
    """Semantic inference for the COCO model on Cityscapes.

    Runs EoMT's semantic inference path (sliding-window) to get 133-class
    per-pixel logits, then sum-aggregates them into the 19-class Cityscapes
    space before argmax. Optional horizontal-flip TTA.
    """
    def _forward(x):
        imgs = [x]
        img_sizes = [x.shape[-2:]]
        crops, origins = model.window_imgs_semantic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], model.img_size, mode="bilinear"
        )
        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )
        logits_coco = model.revert_window_logits_semantic(
            crop_logits, origins, img_sizes
        )[0]
        return coco_logits_to_cityscapes_logits(logits_coco)

    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        x = img.to(f"cuda:{device}")
        logits_cs = _forward(x)
        if tta_hflip:
            logits_flip = _forward(torch.flip(x, dims=[-1]))
            logits_flip = torch.flip(logits_flip, dims=[-1])
            logits_cs = (logits_cs + logits_flip) / 2

    return logits_cs.argmax(dim=0).cpu().numpy()


def infer_panoptic(img, target):
    """Panoptic inference for the COCO model — used for qualitative visualization only."""
    torch.cuda.empty_cache()
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(f"cuda:{device}")]
        img_sizes = [img.shape[-2:] for img in imgs]

        transformed_imgs = model.resize_and_pad_imgs_instance_panoptic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model(transformed_imgs)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], model.img_size, mode="bilinear"
        )
        mask_logits = model.revert_resize_and_pad_logits_instance_panoptic(
            mask_logits, img_sizes
        )
        preds = model.to_per_pixel_preds_panoptic(
            mask_logits,
            class_logits_per_layer[-1],
            model.stuff_classes,
            model.mask_thresh,
            model.overlap_thresh,
        )[0].cpu()

    pred = preds.numpy()
    sem_pred, inst_pred = pred[..., 0], pred[..., 1]
    target_seg = model.to_per_pixel_targets_panoptic([target])[0].cpu().numpy()
    sem_target, inst_target = target_seg[..., 0], target_seg[..., 1]
    return sem_pred, inst_pred, sem_target, inst_target


## Visualization helpers


In [ ]:
def plot_semantic_results(img, pred_array, target_array):
    mapping = create_mapping([pred_array, target_array], IGNORE_INDEX)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img.permute(1, 2, 0).cpu().numpy())
    axes[0].set_title("Image")
    axes[1].imshow(apply_colormap(pred_array, mapping))
    axes[1].set_title("Prediction")
    axes[2].imshow(apply_colormap(target_array, mapping))
    axes[2].set_title("Target")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def _draw_black_border(sem, inst, mapping):
    h, w = sem.shape
    out = np.zeros((h, w, 3))
    for s in np.unique(sem):
        out[sem == s] = mapping[s]
    combined = sem.astype(np.int64) * 100000 + inst.astype(np.int64)
    border = np.zeros((h, w), dtype=bool)
    border[1:, :]  |= combined[1:, :] != combined[:-1, :]
    border[:-1, :] |= combined[1:, :] != combined[:-1, :]
    border[:, 1:]  |= combined[:, 1:] != combined[:, :-1]
    border[:, :-1] |= combined[:, 1:] != combined[:, :-1]
    out[border] = 0
    return out


def plot_panoptic_results(img, sem_pred, inst_pred, sem_target, inst_target):
    all_ids = np.union1d(np.unique(sem_pred), np.unique(sem_target))
    mapping = {
        s: ([0, 0, 0] if s == -1 or s == model.num_classes
            else plt.cm.hsv(i / max(len(all_ids), 1))[:3])
        for i, s in enumerate(all_ids)
    }
    vis_pred   = _draw_black_border(sem_pred,   inst_pred,   mapping)
    vis_target = _draw_black_border(sem_target, inst_target, mapping)
    img_np = img.cpu().numpy().transpose(1, 2, 0) if img.dim() == 3 else img.cpu().numpy()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_np);     axes[0].set_title("Input")
    axes[1].imshow(vis_pred);   axes[1].set_title("Prediction")
    axes[2].imshow(vis_target); axes[2].set_title("Target")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


## Qualitative comparison on a sample image


In [ ]:
# Qualitative visualization on one validation image.
# Cityscapes model -> semantic; COCO model -> panoptic (per the brief).
img, target = data.val_dataloader().dataset[img_idx]

if TASK == "cityscapes":
    pred_array, target_array = infer_semantic(img, target)
    plot_semantic_results(img, pred_array, target_array)
else:
    sem_pred, inst_pred, sem_target, inst_target = infer_panoptic(img, target)
    plot_panoptic_results(img, sem_pred, inst_pred, sem_target, inst_target)


## Quantitative evaluation: mIoU on the full validation set


In [ ]:
from tqdm import tqdm


def evaluate_and_print(model, dataloader, is_coco=False):
    """Per-class and mean IoU on Cityscapes val.

    Standard Cityscapes protocol: ignore_index=255 pixels are excluded from
    both intersection and union; per-class IoU = sum_intersection / sum_union
    across the dataset; mIoU is the mean over classes with non-empty union.
    """
    intersections = np.zeros(19)
    unions = np.zeros(19)

    for batch in tqdm(dataloader, desc=f"Eval [{TASK}]"):
        img, target = batch
        if isinstance(img, list):    img    = img[0]
        if isinstance(target, list): target = target[0]

        if is_coco:
            pred_array = infer_semantic_from_coco(img)
            target_array = model.to_per_pixel_targets_semantic(
                [target], IGNORE_INDEX
            )[0].numpy()
        else:
            pred_array, target_array = infer_semantic(img, target)

        valid_mask = (target_array != IGNORE_INDEX)
        for cls in range(19):
            pred_mask   = (pred_array == cls)
            target_mask = (target_array == cls)
            intersections[cls] += (pred_mask & target_mask & valid_mask).sum()
            unions[cls]        += ((pred_mask | target_mask) & valid_mask).sum()

    print(f"\n{'Class':<20} {'IoU':>7}")
    print("-" * 28)
    ious = []
    for cls in range(19):
        if unions[cls] == 0:
            continue
        iou = intersections[cls] / unions[cls]
        ious.append(iou)
        print(f"{CITYSCAPES_NAMES[cls]:<20} {iou:>7.4f}")
    print("-" * 28)
    miou = float(np.mean(ious)) if ious else 0.0
    print(f"{'mIoU':<20} {miou:>7.4f}")
    return miou


evaluate_and_print(model, data.val_dataloader(), is_coco=(TASK == "coco"))
